# Fine-tuning medical model (QLoRA)

Notebook minimal pour le hackathon: installation, chargement des donnees, entrainement, evaluation, export adapter.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl

In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
from pathlib import Path

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
TRAIN_FILE = "/content/medical_clean_train.jsonl"
VAL_FILE = "/content/medical_clean_val.jsonl"
OUTPUT_DIR = "/content/medical_lora_adapter"

print(torch.cuda.is_available())

## Import des donnees

Option 1: uploader les JSONL dans Colab (`/content/`).  
Option 2: monter Google Drive et adapter les chemins.

In [ ]:
dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})
dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    msgs = example["messages"]
    user = msgs[0]["content"]
    assistant = msgs[1]["content"]
    text = f"<|user|>\n{user}<|end|>\n<|assistant|>\n{assistant}<|end|>"
    return {"text": text}

dataset = dataset.map(format_example)

def tok(batch):
    out = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)
    out["labels"] = out["input_ids"].copy()
    return out

tok_ds = dataset.map(tok, batched=True, remove_columns=dataset["train"].column_names)
tok_ds

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    eval_steps=100,
    evaluation_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds["train"],
    eval_dataset=tok_ds["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter sauvegarde dans:", OUTPUT_DIR)

with open(f"{OUTPUT_DIR}/eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print("Metriques ecrites dans eval_metrics.json")